### import

In [123]:
import os, re, json, sqlite3
from pathlib import Path
from collections import defaultdict
 
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split

### configs

In [ ]:
JSON_PATH    = "DATA.json"
DATASET_ROOT = "dataset"

TUMOR_TYPES = [
    "Astrocytoma", "Ependymoma", "Glioma", "Hemangiopericytoma",
    "Meningioma", "Neurocytoma", "Normal", "Oligodendroglioma",
    "Other", "Schwannoma",
]
WEIGHTINGS = ["T1", "T1C+", "T2"]
 
TUMOR2IDX   = {t: i for i, t in enumerate(TUMOR_TYPES)}
WEIGHT2IDX  = {w: i for i, w in enumerate(WEIGHTINGS)}
 
IMG_SIZE      = 224   # resize from 512; standard for pretrained backbones
BATCH_SIZE    = 32
LR            = 1e-3
LR_FINETUNE   = 3e-5  # conservative LR when backbone is unfrozen to slow overfitting
EPOCHS        = 50
FREEZE_EPOCHS = 10    # longer frozen phase: let heads stabilize before touching backbone
PATIENCE      = 10    # early stopping patience
MIXUP_ALPHA   = 0.3   # MixUp interpolation strength; 0.0 disables it
MAX_IMGS_PER_PATIENT = 30
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DB_PATH       = "brain_tumor.db"

### build db

In [125]:

def build_database(json_path: str, db_path: str = DB_PATH):
    """
    Reads DATA.json and writes every image record into a SQLite database.
 
    Schema
    ------
    images (
        id          INTEGER PRIMARY KEY,
        file_key    TEXT UNIQUE,          -- original JSON key (path within dataset)
        class_folder TEXT,               -- e.g. 'Astrocytoma T1'
        filename    TEXT,                -- e.g. 'T1 - Anaplastic astrocytoma ... 001.jpg'
        tumor_type  TEXT,                -- e.g. 'Astrocytoma'
        weighting   TEXT,                -- e.g. 'T1'
        case_name   TEXT,                -- e.g. 'Anaplastic astrocytoma (pineal region)'
        point_x     INTEGER,
        point_y     INTEGER,
        width       INTEGER,
        height      INTEGER,
        description TEXT,
        class       TEXT
    )
    """
    con = sqlite3.connect(db_path)
    cur = con.cursor()
    cur.execute("DROP TABLE IF EXISTS images")
    cur.execute("""
        CREATE TABLE images (
            id           INTEGER PRIMARY KEY AUTOINCREMENT,
            file_key     TEXT UNIQUE,
            class_folder TEXT,
            filename     TEXT,
            tumor_type   TEXT,
            weighting    TEXT,
            case_name    TEXT,
            point_x      INTEGER,
            point_y      INTEGER,
            width        INTEGER,
            height       INTEGER,
            description  TEXT,
            class        TEXT
        )
    """)
 
    with open(json_path) as f:
        meta = json.load(f)
 
    rows = []
    for key, info in meta.items():
        parts        = key.replace('\\', '/').split('/', 1)
        class_folder = parts[0]
        filename     = parts[1]
        tumor, weighting = parse_class_label(info['class'])
        case_name        = extract_case_id(filename)
        rows.append((
            key, class_folder, filename,
            tumor, weighting, case_name,
            info['point']['x'], info['point']['y'],
            info['width'], info['height'],
            info.get('description', ''),
            info['class'],
        ))
 
    cur.executemany("""
        INSERT OR IGNORE INTO images
            (file_key, class_folder, filename, tumor_type, weighting, case_name,
             point_x, point_y, width, height, description, class)
        VALUES (?,?,?,?,?,?,?,?,?,?,?,?)
    """, rows)
    con.commit()
    con.close()
    print(f"[DB] Imported {len(rows)} records → {db_path}")
 
 
def load_from_db(db_path: str = DB_PATH) -> pd.DataFrame:
    """Query all image records from SQLite into a DataFrame."""
    con = sqlite3.connect(db_path)
    df  = pd.read_sql_query("SELECT * FROM images", con)
    con.close()
    return df

### eda

In [126]:
def run_eda(df: pd.DataFrame):
    sep = "=" * 60
    print(f"\n{sep}\nEXPLORATORY DATA ANALYSIS\n{sep}")
 
    print(f"\nTotal images   : {len(df)}")
    print(f"Unique patients: {df.groupby(['tumor_type','case_name']).ngroups}")
    print(f"Columns        : {list(df.columns)}")
    print(f"Missing values :\n{df.isnull().sum()[df.isnull().sum() > 0].to_string() or '  none'}")
 
    print(f"\n--- Images per tumor type ---")
    print(df['tumor_type'].value_counts().to_string())
 
    print(f"\n--- Images per MRI weighting ---")
    print(df['weighting'].value_counts().to_string())
 
    print(f"\n--- Images per full class (tumor × weighting) ---")
    print(df['class'].value_counts().to_string())
 
    print(f"\n--- Lesion point_x stats ---")
    print(df['point_x'].describe().to_string())
    print(f"\n--- Lesion point_y stats ---")
    print(df['point_y'].describe().to_string())
 
    # Outlier check: points outside image bounds
    oob = df[(df['point_x'] > df['width']) | (df['point_y'] > df['height'])]
    print(f"\nOut-of-bounds lesion points: {len(oob)}")
 
    # Cases with all 3 weightings vs fewer
    wcount = df.groupby(['tumor_type','case_name'])['weighting'].nunique()
    print(f"\n--- Weighting coverage per patient ---")
    print(wcount.value_counts().rename_axis('# weightings').reset_index(name='# patients').to_string(index=False))
 
    # Class imbalance ratio
    counts = df['tumor_type'].value_counts()
    print(f"\nImbalance ratio (max/min tumor type): {counts.max()/counts.min():.2f}x")
    print(f"  Most common : {counts.idxmax()} ({counts.max()})")
    print(f"  Least common: {counts.idxmin()} ({counts.min()})")
 
    print(f"\n{sep}\nEDA COMPLETE\n{sep}\n")

### feature eng

In [127]:
def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds derived columns to the metadata DataFrame.
    These features are available for any downstream analysis; the CNN
    uses tumor_idx and weight_idx encoded below.
    """
    # Numeric label columns
    df['tumor_idx']  = df['tumor_type'].map(TUMOR2IDX)
    df['weight_idx'] = df['weighting'].map(WEIGHT2IDX)
 
    # Normalised lesion position (0–1 relative to image size)
    df['point_x_norm'] = df['point_x'] / df['width']
    df['point_y_norm'] = df['point_y'] / df['height']
 
    # Distance of lesion centre from image centre
    df['lesion_offset'] = np.sqrt(
        (df['point_x'] - df['width']  / 2) ** 2 +
        (df['point_y'] - df['height'] / 2) ** 2
    )
 
    # Boolean: is this a Normal (no lesion) scan?
    df['is_normal'] = (df['tumor_type'] == 'Normal').astype(int)
 
    # Description length as a proxy for annotation richness
    df['desc_length'] = df['description'].str.len().fillna(0).astype(int)
 
    # Ordinal weighting index (T1=0, T1C+=1, T2=2) — already in weight_idx,
    # but make it explicit as an ordinal for clarity
    df['weighting_ordinal'] = df['weight_idx']   # alias for readability
 
    print(f"[FE] Added columns: point_x_norm, point_y_norm, lesion_offset, "
          f"is_normal, desc_length, tumor_idx, weight_idx, weighting_ordinal")
    return df

### parse dataset

In [128]:
def extract_case_id(filename: str) -> str:
    """
    filename: bare filename inside the class folder, e.g.
              'T1 - Anaplastic astrocytoma (pineal region) 001.jpg'
    Returns:  'Anaplastic astrocytoma (pineal region)'
    """
    name = re.sub(r'^T1C\+\s*[-–]\s*|^T1\s*[-–]\s*|^T2\s*[-–]\s*', '', filename)
    name = re.sub(r'\s*\d+\.jpg$', '', name, flags=re.IGNORECASE).strip()
    return name
 
 
def parse_class_label(class_str: str):
    """
    'Astrocytoma T1C+' → tumor='Astrocytoma', weighting='T1C+'
    Handles the ordering: try T1C+ first, then T2, then T1.
    """
    for w in ["T1C+", "T2", "T1"]:
        if class_str.endswith(w):
            tumor = class_str[: -len(w)].strip()
            return tumor, w
    raise ValueError(f"Cannot parse class: {class_str}")
 
 
def build_disk_index(dataset_root: str) -> dict:
    """
    Walk dataset_root once and build:
        (class_folder, nfc_filename) -> absolute path
    Using on-disk names avoids any JSON <-> filesystem encoding mismatch.
    """
    import unicodedata
    index = {}
    for folder in os.listdir(dataset_root):
        folder_path = os.path.join(dataset_root, folder)
        if not os.path.isdir(folder_path):
            continue
        for fname in os.listdir(folder_path):
            nfc_key = unicodedata.normalize('NFC', fname).strip()
            index[(folder, nfc_key)] = os.path.join(folder_path, fname)
    return index
 
 
def resolve_path(disk_index: dict, class_folder: str, filename: str) -> str:
    """
    Look up the real on-disk path, tolerating:
      - Unicode normalization differences (NFC/NFKC)
      - UTF-8 bytes decoded as cp1252 (mojibake) e.g. en-dash stored as â€"
      - ASCII hyphen substitution for Unicode dashes
    """
    import unicodedata
 
    def candidates(name):
        yield name
        # mojibake: encode as utf-8 then decode as cp1252 (matches Windows mis-saved filenames)
        try:
            yield name.encode('utf-8').decode('cp1252')
        except (UnicodeDecodeError, UnicodeEncodeError):
            pass
        # dash -> hyphen fallback
        for dash in ('–', '—', '‒'):
            if dash in name:
                yield name.replace(dash, '-')
 
    for form in ('NFC', 'NFKC'):
        norm = unicodedata.normalize(form, filename).strip()
        for candidate in candidates(norm):
            if (class_folder, candidate) in disk_index:
                return disk_index[(class_folder, candidate)]
 
    return os.path.join(class_folder, filename)
 
 
def build_patient_split(dataset_root: str, db_path: str = DB_PATH, seed: int = 42):
    """
    Loads records from SQLite, groups by patient, splits 70/15/15.
    - Stratified by tumor type so rare classes (8 patients) are represented in all splits.
    - Caps images per patient at MAX_IMGS_PER_PATIENT to prevent high-slice patients dominating.
    Returns three lists of dicts for the DataLoader.
    """
    import random
    rng = random.Random(seed)
 
    df = load_from_db(db_path)
    df = feature_engineering(df)
 
    disk_index = build_disk_index(dataset_root)
 
    # Build patient_images map with per-patient cap
    patient_images = defaultdict(list)
    for _, row in df.iterrows():
        patient_id = (row['tumor_type'], row['case_name'])
        abs_path   = resolve_path(disk_index, row['class_folder'], row['filename'])
        patient_images[patient_id].append({
            'path':       abs_path,
            'tumor_idx':  int(row['tumor_idx']),
            'weight_idx': int(row['weight_idx']),
        })
 
    # Apply per-patient image cap (shuffle first so cap is random, not first-N slices)
    for pid in patient_images:
        imgs = patient_images[pid]
        rng.shuffle(imgs)
        patient_images[pid] = imgs[:MAX_IMGS_PER_PATIENT]
 
    # Group patient_ids by tumor type for stratified split
    tumor_to_patients = defaultdict(list)
    for pid in patient_images:
        tumor_to_patients[pid[0]].append(pid)
 
    train_ids, val_ids, test_ids = [], [], []
    for tumor, pids in tumor_to_patients.items():
        rng.shuffle(pids)
        n = len(pids)
        n_test = max(1, round(n * 0.15))
        n_val  = max(1, round(n * 0.15))
        test_ids  += pids[:n_test]
        val_ids   += pids[n_test:n_test + n_val]
        train_ids += pids[n_test + n_val:]
        print(f"  {tumor:<22} total={n:3d} | train={len(pids[n_test+n_val:]):2d} val={n_val} test={n_test}")
 
    def flatten(ids):
        records = []
        for pid in ids:
            records.extend(patient_images[pid])
        return records
 
    train_records = flatten(train_ids)
    val_records   = flatten(val_ids)
    test_records  = flatten(test_ids)
 
    print(f"Patients  — train: {len(train_ids)}, val: {len(val_ids)}, test: {len(test_ids)}")
    print(f"Images    — train: {len(train_records)}, val: {len(val_records)}, test: {len(test_records)}")
 
    assert not (set(train_ids) & set(val_ids)),   "Train/Val patient leak!"
    assert not (set(train_ids) & set(test_ids)),  "Train/Test patient leak!"
    assert not (set(val_ids)   & set(test_ids)),  "Val/Test patient leak!"
 
    return train_records, val_records, test_records

### dataset

In [129]:
class BrainMRIDataset(Dataset):
    def __init__(self, records, transform=None):
        self.records   = records
        self.transform = transform
 
    def __len__(self):
        return len(self.records)
 
    def __getitem__(self, idx):
        rec  = self.records[idx]
        img  = Image.open(rec['path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, rec['tumor_idx'], rec['weight_idx']
 
 
def get_transforms(train: bool):
    # ImageNet stats — required because backbone is pretrained on ImageNet
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]
    if train:
        return transforms.Compose([
            transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
            transforms.RandomCrop(IMG_SIZE),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(20),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.1),
            transforms.RandomGrayscale(p=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
            transforms.RandomErasing(p=0.3, scale=(0.02, 0.15)),  # Cutout-style
        ])
    return transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

### model

In [130]:

class ConvBlock(nn.Module):
    """Conv → BN → ReLU → MaxPool"""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),         # halve spatial dims
        )
 
    def forward(self, x):
        return self.block(x)
 
 
class MultiTaskCNN(nn.Module):
    """
    Backbone: ResNet-18 pretrained on ImageNet, final FC removed.
    Feature dim: 512.
    Phase 1 (epochs 1..FREEZE_EPOCHS): backbone frozen, only heads trained.
    Phase 2 (epochs FREEZE_EPOCHS+1..end): full network fine-tuned at lower LR.
    Head 1: tumor type  (10-class)
    Head 2: MRI weighting (3-class)
    """
    def __init__(self, num_tumor=10, num_weighting=3, dropout=0.5):
        super().__init__()
        base         = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        feat_dim     = base.fc.in_features  # 512
        base.fc      = nn.Identity()        # remove classification head
        self.backbone = base
 
        self.head_tumor = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feat_dim, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, num_tumor),
        )
        self.head_weight = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feat_dim, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, num_weighting),
        )
 
    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = False
 
    def unfreeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = True
 
    def forward(self, x):
        feats         = self.backbone(x)         # (B, 512)
        logits_tumor  = self.head_tumor(feats)   # (B, 10)
        logits_weight = self.head_weight(feats)  # (B, 3)
        return logits_tumor, logits_weight   



### train 

In [131]:
def compute_class_weights(records, num_classes, label_key):
    counts = np.zeros(num_classes, dtype=np.float32)
    for r in records:
        counts[r[label_key]] += 1
    weights = counts.sum() / (num_classes * counts)
    return torch.tensor(weights, dtype=torch.float32).to(DEVICE)
 
 
def mixup_batch(imgs, t_labels, w_labels, alpha=MIXUP_ALPHA):
    """
    MixUp: interpolate pairs of images and produce soft labels.
    Returns mixed images and two sets of (label, lambda) for loss computation.
    Only applied during training (caller's responsibility).
    """
    if alpha <= 0:
        return imgs, t_labels, t_labels, w_labels, w_labels, 1.0
    lam = np.random.beta(alpha, alpha)
    B   = imgs.size(0)
    idx = torch.randperm(B, device=imgs.device)
    mixed = lam * imgs + (1 - lam) * imgs[idx]
    return mixed, t_labels, t_labels[idx], w_labels, w_labels[idx], lam
 
 
def run_epoch(model, loader, criterion_t, criterion_w, optimizer, train: bool):
    model.train() if train else model.eval()
    total_loss = tumor_correct = weight_correct = total = 0
 
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, t_labels, w_labels in loader:
            imgs     = imgs.to(DEVICE)
            t_labels = t_labels.to(DEVICE)
            w_labels = w_labels.to(DEVICE)
 
            if train:
                imgs, t_a, t_b, w_a, w_b, lam = mixup_batch(imgs, t_labels, w_labels)
                logits_t, logits_w = model(imgs)
                loss_t = lam * criterion_t(logits_t, t_a) + (1 - lam) * criterion_t(logits_t, t_b)
                loss_w = lam * criterion_w(logits_w, w_a) + (1 - lam) * criterion_w(logits_w, w_b)
                # accuracy against primary label only (standard MixUp reporting)
                tumor_correct  += (logits_t.argmax(1) == t_a).sum().item()
                weight_correct += (logits_w.argmax(1) == w_a).sum().item()
            else:
                logits_t, logits_w = model(imgs)
                loss_t = criterion_t(logits_t, t_labels)
                loss_w = criterion_w(logits_w, w_labels)
                tumor_correct  += (logits_t.argmax(1) == t_labels).sum().item()
                weight_correct += (logits_w.argmax(1) == w_labels).sum().item()
 
            loss = loss_t + 0.5 * loss_w
 
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
 
            total_loss += loss.item() * imgs.size(0)
            total      += imgs.size(0)
 
    return total_loss / total, tumor_correct / total, weight_correct / total
 
 
def train(dataset_root: str):
    # --- Data ---
    train_rec, val_rec, test_rec = build_patient_split(dataset_root)
 
    train_ds = BrainMRIDataset(train_rec, get_transforms(train=True))
    val_ds   = BrainMRIDataset(val_rec,   get_transforms(train=False))
    test_ds  = BrainMRIDataset(test_rec,  get_transforms(train=False))
 
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
 
    # Class-weighted CE to handle imbalance (Hemangiopericytoma: 118 vs Meningioma T1C+: 977)
    w_tumor  = compute_class_weights(train_rec, 10, 'tumor_idx')
    w_weight = compute_class_weights(train_rec, 3,  'weight_idx')
    criterion_t = nn.CrossEntropyLoss(weight=w_tumor)
    criterion_w = nn.CrossEntropyLoss(weight=w_weight)
 
    # --- Model ---
    model = MultiTaskCNN().to(DEVICE)
 
    # Phase 1: freeze backbone, train heads only
    model.freeze_backbone()
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
 
    best_val_loss = float('inf')
    patience_counter = 0
 
    print(f"\nTraining on {DEVICE}\n{'='*60}")
    print(f"Phase 1: backbone frozen for {FREEZE_EPOCHS} epochs")
    for epoch in range(1, EPOCHS + 1):
 
        # Phase 2: unfreeze backbone after FREEZE_EPOCHS
        if epoch == FREEZE_EPOCHS + 1:
            print(f"\nPhase 2: unfreezing backbone at epoch {epoch}, LR → {LR_FINETUNE}")
            model.unfreeze_backbone()
            optimizer = torch.optim.Adam(model.parameters(), lr=LR_FINETUNE, weight_decay=1e-4)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=EPOCHS - FREEZE_EPOCHS
            )
 
        tr_loss, tr_tacc, tr_wacc = run_epoch(model, train_loader, criterion_t, criterion_w, optimizer, train=True)
        va_loss, va_tacc, va_wacc = run_epoch(model, val_loader,   criterion_t, criterion_w, optimizer, train=False)
        scheduler.step()
 
        print(
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"Train Loss {tr_loss:.4f} | Tumor Acc {tr_tacc:.3f} | Weight Acc {tr_wacc:.3f} || "
            f"Val Loss {va_loss:.4f} | Tumor Acc {va_tacc:.3f} | Weight Acc {va_wacc:.3f}"
        )
 
        if va_loss < best_val_loss:
            best_val_loss = va_loss
            patience_counter = 0
            torch.save(model.state_dict(), "best_model.pt")
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"\nEarly stopping at epoch {epoch} (no val improvement for {PATIENCE} epochs)")
                break
 
    # --- Test ---
    print("\nLoading best checkpoint for test evaluation…")
    model.load_state_dict(torch.load("best_model.pt", map_location=DEVICE))
    te_loss, te_tacc, te_wacc = run_epoch(model, test_loader, criterion_t, criterion_w, optimizer, train=False)
    print(f"Test Loss {te_loss:.4f} | Tumor Acc {te_tacc:.3f} | Weighting Acc {te_wacc:.3f}")
 
    return model

### do train

In [ ]:
build_database(JSON_PATH, DB_PATH)


[DB] Imported 11300 records → brain_tumor.db


In [ ]:
df = load_from_db(DB_PATH)
run_eda(df)



EXPLORATORY DATA ANALYSIS

Total images   : 11300
Unique patients: 274
Columns        : ['id', 'file_key', 'class_folder', 'filename', 'tumor_type', 'weighting', 'case_name', 'point_x', 'point_y', 'width', 'height', 'description', 'class']
Missing values :
Series([], )

--- Images per tumor type ---
tumor_type
Meningioma            2071
Other                 1533
Glioma                1476
Schwannoma            1236
Astrocytoma           1118
Normal                1058
Ependymoma            1037
Neurocytoma            618
Hemangiopericytoma     603
Oligodendroglioma      550

--- Images per MRI weighting ---
weighting
T1C+    4678
T1      3646
T2      2976

--- Images per full class (tumor × weighting) ---
class
Meningioma T1C+            977
Other T1C+                 756
Meningioma T1              636
Schwannoma T1C+            564
Glioma T1C+                549
Glioma T1                  522
Meningioma T2              458
Astrocytoma T1C+           441
Normal T1                  41

In [134]:

# Section 2: feature engineering (enriches DataFrame; labels used by DataLoader)
# (feature_engineering is called inside build_patient_split automatically)

# Sections 3-5: train
train(DATASET_ROOT)

[FE] Added columns: point_x_norm, point_y_norm, lesion_offset, is_normal, desc_length, tumor_idx, weight_idx, weighting_ordinal
  Astrocytoma            total= 40 | train=28 val=6 test=6
  Ependymoma             total= 20 | train=14 val=3 test=3
  Glioma                 total= 40 | train=28 val=6 test=6
  Hemangiopericytoma     total=  8 | train= 6 val=1 test=1
  Meningioma             total= 52 | train=36 val=8 test=8
  Neurocytoma            total=  8 | train= 6 val=1 test=1
  Normal                 total= 13 | train= 9 val=2 test=2
  Oligodendroglioma      total= 17 | train=11 val=3 test=3
  Other                  total= 44 | train=30 val=7 test=7
  Schwannoma             total= 32 | train=22 val=5 test=5
Patients  — train: 190, val: 42, test: 42
Images    — train: 4033, val: 857, test: 837

Training on cuda
Phase 1: backbone frozen for 10 epochs
Epoch 01/50 | Train Loss 2.6913 | Tumor Acc 0.127 | Weight Acc 0.483 || Val Loss 2.4040 | Tumor Acc 0.224 | Weight Acc 0.653
Epoch 02/50 |

MultiTaskCNN(
  (backbone): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-0